# LaBraM Pretrained + LoRA

This notebook loads the Braindecode pretrained LaBraM checkpoint through `from_pretrained`, adapts the model to the current channel layout with interpolation, and injects LoRA adapters into the attention blocks.


In [2]:
import sys
import subprocess
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
%pip install -q mne
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Define IN_COLAB based on sys.modules
IN_COLAB = "google.colab" in sys.modules

# Mount Google Drive if in Colab, before defining SRC_DIR
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        # This case should ideally not happen if "google.colab" is in sys.modules
        # but if it does, it means drive module itself is not available
        print("Warning: Running in Colab, but google.colab.drive module not found.")

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (
    Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if IN_COLAB
    else NOTEBOOK_DIR.parent.parent / "src"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import BinaryTaskConfig, RawEpochSpec, detect_project_paths, load_raw_epoch_dataset, get_task_config
from br_analysis.pretrained import BackboneClassifier, prepare_lora_backbone, load_backbone_weights
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
)
from br_analysis.torch_models import count_trainable_parameters

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")


^C
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'pandas'

## Task And Environment


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "braindecode[hub]"], check=True)

TASK_TAG = "c_vs_ip_labram_pretrained_lora"
paths = detect_project_paths("c_vs_i_labram_pretrained_lora", in_colab=IN_COLAB)
SELECTED_SUBJECTS_CSV = paths.analysis_root / "01_exploracio" / "tables" / "subjectes_seleccionats.csv"
MAX_SUBJECTS = 24

TASK = get_task_config("c_vs_ip", paths.analysis_root, name=TASK_TAG, analysis_window=(0.0, 1.5))

print(f"Projecte: {paths.project_dir}")
print(f"Resultats: {paths.analysis_dir}")


## Load Raw Epochs


In [ ]:
spec = RawEpochSpec(picks=None, bandpass=(1.0, 40.0), resample_sfreq=128.0)
dataset = load_raw_epoch_dataset(paths, TASK, spec)
X = np.asarray(dataset.X, dtype=np.float32)
y = np.asarray(dataset.y, dtype=np.int64)
subjects = np.asarray(dataset.subjects, dtype=object)
SUBJECT_ID = dataset.subject_ids[0]  # Change this value to train/evaluate another subject.
chs_info = dataset.chs_info
sfreq = dataset.sfreq or (1.0 / float(np.median(np.diff(dataset.times))))

# Padding to 3000 samples
TARGET_LEN = 3000
if X.shape[-1] < TARGET_LEN:
    X = np.pad(X, ((0, 0), (0, 0), (0, TARGET_LEN - X.shape[-1])), mode="constant")
else:
    X = X[:, :, :TARGET_LEN]

print(f"X shape: {X.shape}")
print(f"Classes: {np.bincount(y)}")

plt.figure(figsize=(5, 4))
sns.countplot(x=np.where(y == 0, TASK.class_0, TASK.class_1))
plt.title("Class distribution")
plt.xlabel("Class")
plt.tight_layout()
plt.show()

## Build Pretrained Model


In [ ]:
from braindecode.models import InterpolatedLaBraM
import mne

ch_names = [ch["ch_name"] for ch in chs_info]
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
montage = mne.channels.make_standard_montage("standard_1005")
info.set_montage(montage)

model = InterpolatedLaBraM.from_pretrained(
    "braindecode/labram-pretrained",
    chs_info=info["chs"],
    n_times=3000,  # Adjusted to 3000 to match checkpoint embeddings
    n_outputs=2,
    patch_size=200,
    strict=False,
)

prepare_lora_backbone(
    model,
    include_substrings=("attn",),
    exclude_substrings=("head", "classifier", "interpolation_layer"),
    r=8,
    alpha=16.0,
    dropout=0.05,
    train_classifier=True,
)

if hasattr(model, "get_classifier"):
    for p in model.get_classifier().parameters():
        p.requires_grad = True

trainable, total = count_trainable_parameters(model)
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## Train


In [ ]:
def make_model():
    from braindecode.models import InterpolatedLaBraM

    # Re-generate the correct channel info with montage for each model instance
    ch_names = [ch["ch_name"] for ch in chs_info]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    montage = mne.channels.make_standard_montage("standard_1005")
    info.set_montage(montage)

    model = InterpolatedLaBraM.from_pretrained(
        "braindecode/labram-pretrained",
        chs_info=info["chs"],
        n_times=3000,  # Must match the padded X length
        n_outputs=2,
        patch_size=200,
        strict=False,
    )

    prepare_lora_backbone(
        model,
        include_substrings=("attn",),
        exclude_substrings=("head", "classifier", "interpolation_layer"),
        r=8,
        alpha=16.0,
        dropout=0.05,
        train_classifier=True,
    )

    if hasattr(model, "get_classifier"):
        for p in model.get_classifier().parameters():
            p.requires_grad = True

    return model


train_cfg = TorchTrainConfig(
    epochs=30,
    batch_size=4,
    lr=2e-4,
    weight_decay=1e-4,
    patience=6,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
    progress=True,
)

metrics, history = cross_validate_torch_model(X, y, make_model, train_cfg, subjects=subjects, subject_id=SUBJECT_ID, plot_roc_pr=True, figures_dir=paths.figures_dir, task_tag=TASK_TAG)
metrics_path = paths.tables_dir / f"cv_metrics_{TASK_TAG}.csv"
history_path = paths.tables_dir / f"training_history_{TASK_TAG}.csv"
metrics.to_csv(metrics_path, index=False)
history.to_csv(history_path, index=False)
print(f"Saved metrics: {metrics_path}")
print(f"Saved history: {history_path}")
metrics

## Visualize Training


In [ ]:
plot_fold_metrics(metrics, output_file=paths.figures_dir / f"fold_metrics_{TASK_TAG}.png")
plot_torch_history(history, output_file=paths.figures_dir / f"training_history_{TASK_TAG}.png")

summary = metrics.drop(columns=["fold"]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
summary_path = paths.tables_dir / f"summary_{TASK_TAG}.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary